# MMLU Inference (parameterized)

Runs any HuggingFace causal-LM on MMLU and saves token-level logprobs over A/B/C/D.

**Setup**:
1. Runtime → Change runtime type → T4 GPU
2. Upload `mmlu.parquet` to `My Drive/Colab Notebooks/`
3. Set HF token in Cell 3
4. Pick a model in Cell 4 (uncomment exactly one `MODEL_CONFIG` block)

**Output**: `My Drive/Colab Notebooks/{slug}_responses.parquet`

**Two regimes**:
- `instruct`: zero-shot, chat template + `"The answer is "` pre-fill (matches existing Llama-Instruct collection)
- `base`: 5-shot, completion-style, dev examples drawn subject-matched from MMLU dev split (lm-eval-harness convention)

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Install dependencies
!pip install -q transformers accelerate bitsandbytes pandas pyarrow tqdm datasets

In [ ]:
# Cell 3: HuggingFace token
# Get one from https://huggingface.co/settings/tokens
# For Llama-3 base/instruct, you must accept the license at https://huggingface.co/meta-llama/Meta-Llama-3-8B
import os
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"  # replace this

In [ ]:
# Cell 4: MODEL_CONFIG — uncomment exactly one block.
# Suggested order: 1.5B (smoke test) → Llama base → Qwen 7B → Qwen 14B.

# --- Qwen-2.5-1.5B-Instruct (smoke test, ~30 min on T4) ---
MODEL_CONFIG = {
    "model_id":   "Qwen/Qwen2.5-1.5B-Instruct",
    "regime":     "instruct",
    "slug":       "qwen25_1_5b_instruct",
    "batch_size": 16,
}

# --- Llama-3-8B base (5-shot, completion-style) ---
# MODEL_CONFIG = {
#     "model_id":   "meta-llama/Meta-Llama-3-8B",
#     "regime":     "base",
#     "slug":       "llama3_8b_base",
#     "batch_size": 4,   # 5-shot prompts are long; smaller batch needed
# }

# --- Qwen-2.5-7B-Instruct (cross-family match to Llama-Instruct) ---
# MODEL_CONFIG = {
#     "model_id":   "Qwen/Qwen2.5-7B-Instruct",
#     "regime":     "instruct",
#     "slug":       "qwen25_7b_instruct",
#     "batch_size": 8,
# }

# --- Qwen-2.5-14B-Instruct (largest scale point) ---
# MODEL_CONFIG = {
#     "model_id":   "Qwen/Qwen2.5-14B-Instruct",
#     "regime":     "instruct",
#     "slug":       "qwen25_14b_instruct",
#     "batch_size": 4,
# }

# --- Reference: Llama-3-8B-Instruct (already collected; here for completeness) ---
# MODEL_CONFIG = {
#     "model_id":   "meta-llama/Meta-Llama-3-8B-Instruct",
#     "regime":     "instruct",
#     "slug":       "llama_instruct",
#     "batch_size": 8,
# }

assert MODEL_CONFIG["regime"] in ("instruct", "base")
print(f"Selected: {MODEL_CONFIG['model_id']}  ({MODEL_CONFIG['regime']})")
print(f"Output slug: {MODEL_CONFIG['slug']}")

In [ ]:
# Cell 5: Paths
import pathlib

DRIVE_DIR = pathlib.Path("/content/drive/MyDrive/Colab Notebooks")
IN  = DRIVE_DIR / "mmlu.parquet"
OUT = DRIVE_DIR / f"{MODEL_CONFIG['slug']}_responses.parquet"

assert IN.exists(), f"mmlu.parquet not found at {IN} — upload it to Drive first"
print(f"Input:  {IN}")
print(f"Output: {OUT}")

In [ ]:
# Cell 6: GPU check
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 7: Load tokenizer + model (4-bit on T4)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"Loading tokenizer: {MODEL_CONFIG['model_id']}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CONFIG["model_id"], token=os.environ["HF_TOKEN"], padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in 4-bit (this may take a few minutes)...")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_CONFIG["model_id"],
    token=os.environ["HF_TOKEN"],
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("Model loaded.")

In [ ]:
# Cell 8: Few-shot examples (only used when regime == 'base')
# MMLU dev split has exactly 5 examples per subject by design.
import pandas as pd
from datasets import load_dataset

if MODEL_CONFIG["regime"] == "base":
    print("Loading MMLU dev split for 5-shot examples...")
    dev_ds = load_dataset("cais/mmlu", "all", split="dev")
    dev_df = dev_ds.to_pandas()
    FEWSHOT = {subj: g.to_dict("records") for subj, g in dev_df.groupby("subject")}
    counts = {s: len(v) for s, v in FEWSHOT.items()}
    print(f"Loaded {len(dev_df)} dev examples across {len(FEWSHOT)} subjects.")
    assert all(c == 5 for c in counts.values()), f"Expected 5 per subject, got: {counts}"
else:
    FEWSHOT = None
    print("Instruct regime — no few-shot examples needed.")

In [ ]:
# Cell 9: Helpers (prompt builders, token ID resolution, batched query)
from tqdm.notebook import tqdm

ANSWER_TOKENS = ["A", "B", "C", "D"]


def build_prompt_instruct(row):
    """Zero-shot, chat template, pre-fill 'The answer is ' to score next token."""
    choices = row["choices"]
    user_msg = (
        f"Question: {row['question']}\n\n"
        f"Choices:\nA. {choices[0]}\nB. {choices[1]}\nC. {choices[2]}\nD. {choices[3]}"
    )
    messages = [
        {"role": "system", "content": "Answer the following multiple choice question with a single letter: A, B, C, or D."},
        {"role": "user", "content": user_msg},
    ]
    base = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return base + "The answer is "


def build_prompt_base(row):
    """5-shot completion-style prompt (lm-eval-harness/HELM convention for MMLU)."""
    subject = row["subject"]
    examples = FEWSHOT[subject]
    subject_label = subject.replace("_", " ")
    blocks = [f"The following are multiple choice questions (with answers) about {subject_label}."]
    for ex in examples:
        ex_letter = ANSWER_TOKENS[ex["answer"]]
        blocks.append(
            f"{ex['question']}\n"
            f"A. {ex['choices'][0]}\n"
            f"B. {ex['choices'][1]}\n"
            f"C. {ex['choices'][2]}\n"
            f"D. {ex['choices'][3]}\n"
            f"Answer: {ex_letter}"
        )
    blocks.append(
        f"{row['question']}\n"
        f"A. {row['choices'][0]}\n"
        f"B. {row['choices'][1]}\n"
        f"C. {row['choices'][2]}\n"
        f"D. {row['choices'][3]}\n"
        f"Answer:"
    )
    return "\n\n".join(blocks)


def build_prompt(row):
    if MODEL_CONFIG["regime"] == "instruct":
        return build_prompt_instruct(row)
    return build_prompt_base(row)


def get_answer_token_ids(tokenizer):
    """Both no-space and leading-space token IDs for each of A/B/C/D — we score whichever is higher."""
    ids = {}
    for tok in ANSWER_TOKENS:
        id_no_space   = tokenizer.encode(tok, add_special_tokens=False)[0]
        id_with_space = tokenizer.encode(f" {tok}", add_special_tokens=False)[0]
        ids[tok] = (id_no_space, id_with_space)
    return ids


def query_batch(model, tokenizer, prompts, token_ids):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    last_logits = outputs.logits[:, -1, :]
    log_probs = torch.log_softmax(last_logits, dim=-1).cpu()
    results = []
    for i in range(len(prompts)):
        row_lp = {}
        for tok, (id1, id2) in token_ids.items():
            row_lp[f"logprob_{tok}"] = max(log_probs[i, id1].item(), log_probs[i, id2].item())
        predicted = max(row_lp, key=lambda k: row_lp[k]).replace("logprob_", "")
        results.append({**row_lp, "predicted": predicted})
    return results

print("Helpers defined.")

In [ ]:
# Cell 10: Tokenization diagnostic — RUN BEFORE FULL INFERENCE
# Verifies the prompt format and that A/B/C/D tokens are correctly identified.
import pandas as pd
import torch

df_test = pd.read_parquet(IN)
row = df_test.iloc[0]
prompt = build_prompt(row)

print("=== PROMPT (truncated if long) ===")
if len(prompt) > 800:
    print(prompt[:400])
    print("\n...[truncated]...\n")
    print(prompt[-400:])
else:
    print(prompt)
print()

token_ids = get_answer_token_ids(tokenizer)
print("=== ANSWER TOKEN IDs ===")
for tok, (id1, id2) in token_ids.items():
    print(f"  '{tok}'   id={id1:6d}  decoded={tokenizer.decode([id1])!r}")
    print(f"  ' {tok}'  id={id2:6d}  decoded={tokenizer.decode([id2])!r}")

inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model(**inputs)
log_probs = torch.log_softmax(outputs.logits[0, -1, :], dim=-1).cpu()

top10 = log_probs.topk(10).indices.tolist()
print("\n=== TOP-10 NEXT TOKENS THE MODEL WANTS TO GENERATE ===")
for tid in top10:
    print(f"  id={tid:6d}  logprob={log_probs[tid]:.3f}  token={tokenizer.decode([tid])!r}")

correct_letter = ANSWER_TOKENS[row["answer"]]
print("\n=== OUR SCORED LOGPROBS ===")
for tok, (id1, id2) in token_ids.items():
    lp = max(log_probs[id1].item(), log_probs[id2].item())
    flag = "  ← correct" if tok == correct_letter else ""
    print(f"  logprob_{tok} = {lp:.3f}{flag}")
print("\nIf top-10 contains 'A'/'B'/'C'/'D' (with or without leading space), the prompt is well-formed.")

In [ ]:
# Cell 11: DELETE existing output before a fresh run
# (skip this cell if you want to resume from a checkpoint)
import pathlib
p = pathlib.Path(str(OUT))
if p.exists():
    p.unlink()
    print(f"Deleted {p}")
else:
    print("No existing file — safe to run inference.")

In [ ]:
# Cell 12: Run inference with checkpointing every 100 questions.
import pandas as pd

df = pd.read_parquet(IN)
if OUT.exists():
    done = pd.read_parquet(OUT)["question_id"].tolist()
    df = df[~df["question_id"].isin(done)]
    print(f"Resuming: {len(df):,} questions remaining")
else:
    print(f"Starting fresh: {len(df):,} questions")

token_ids = get_answer_token_ids(tokenizer)
BATCH_SIZE = MODEL_CONFIG["batch_size"]
SAVE_EVERY = 100
results = []

for i in tqdm(range(0, len(df), BATCH_SIZE)):
    batch = df.iloc[i : i + BATCH_SIZE]
    prompts = [build_prompt(row) for _, row in batch.iterrows()]
    batch_results = query_batch(model, tokenizer, prompts, token_ids)
    for j, (_, row) in enumerate(batch.iterrows()):
        res = batch_results[j]
        res["question_id"] = row["question_id"]
        res["correct"] = res["predicted"] == ANSWER_TOKENS[row["answer"]]
        results.append(res)

    if len(results) >= SAVE_EVERY:
        out_df = pd.DataFrame(results)
        if OUT.exists():
            out_df = pd.concat([pd.read_parquet(OUT), out_df], ignore_index=True)
        out_df.to_parquet(OUT, index=False)
        results = []

if results:
    out_df = pd.DataFrame(results)
    if OUT.exists():
        out_df = pd.concat([pd.read_parquet(OUT), out_df], ignore_index=True)
    out_df.to_parquet(OUT, index=False)

print(f"Done. {len(pd.read_parquet(OUT)):,} total responses saved to {OUT}")

In [ ]:
# Cell 13: Post-inference diagnostic
import pandas as pd
import numpy as np
from scipy.special import softmax

df_out = pd.read_parquet(OUT)
print(f"Model: {MODEL_CONFIG['model_id']}")
print(f"Total responses: {len(df_out):,}")
print(f"Accuracy: {df_out['correct'].mean():.4f}")
print()

print("=== PREDICTION DISTRIBUTION (should be roughly uniform ~0.25 each) ===")
dist = df_out["predicted"].value_counts(normalize=True).sort_index()
for letter, frac in dist.items():
    bar = "#" * int(frac * 40)
    print(f"  {letter}: {frac:.3f}  {bar}")
print()

print("=== MEAN LOGPROBS PER LETTER (large asymmetry suggests positional bias) ===")
for col in ["logprob_A", "logprob_B", "logprob_C", "logprob_D"]:
    print(f"  {col}: {df_out[col].mean():.3f}")
print()

THRESHOLD = -8.0
suspicious = (df_out[["logprob_A","logprob_B","logprob_C","logprob_D"]] < THRESHOLD).all(axis=1)
print("=== NON-LETTER GENERATION ESTIMATE ===")
print(f"  All logprob_ABCD < {THRESHOLD}: {suspicious.sum():,} ({suspicious.mean():.1%})")
print("  High percentages here mean the model often preferred a non-letter token (e.g. number/word).")
print()

logprobs = df_out[["logprob_A","logprob_B","logprob_C","logprob_D"]].values
probs = softmax(logprobs, axis=1)
confidence = probs.max(axis=1)
print("=== CONFIDENCE DISTRIBUTION (renormalized over A/B/C/D) ===")
print(f"  Mean:   {confidence.mean():.3f}")
print(f"  Median: {np.median(confidence):.3f}")